# 51 — Online U20 action-displacement gate analysis

LIBERO-PRO only. Exact-match stock, ungated U20 gradient, gate 0.015, and gate 0.020 by suite/task/init/hash. Every SR and delta uses all matched identities, never only accepted chunks. The two caps were predeclared before collection; this notebook does not optimize a threshold on outcomes.

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from analysis.uncertainty_gradient_gate import (
    gate_telemetry_table, match_action_gate_cohort, paired_effect_table,
    success_table, suite_success_table)
from pnp.store import SupabaseStore

EXPECTED_IDENTITIES = 220
REQUIRE_FULL_COHORT = True  # use False only for a clearly labeled interim preview
OUTPUT = Path('u20_action_gate_pro220_outputs')
OUTPUT.mkdir(exist_ok=True)
store = SupabaseStore()

## Exact matching and primary result

In [ ]:
paired, coverage, historical_coverage = match_action_gate_cohort(
    store, expected_identities=EXPECTED_IDENTITIES,
    require_complete=REQUIRE_FULL_COHORT)
print('FINAL STRICT ANALYSIS' if REQUIRE_FULL_COHORT else 'INTERIM MATCHED PREVIEW')
print(f'{len(paired)} identities in every SR denominator')
display(coverage)
if not REQUIRE_FULL_COHORT:
    print('Preview only: rerun with REQUIRE_FULL_COHORT=True after all four workers finish.')

success = success_table(paired)
effects = paired_effect_table(paired)
print('Success rates on all matched identities')
display(success)
print('Paired SR changes; positive means the named condition is better')
display(effects[['comparison', 'matched_episodes', 'reference_sr_pct',
                 'condition_sr_pct', 'condition_minus_reference_pp',
                 'delta_ci_low_pp', 'delta_ci_high_pp', 'F_to_S', 'S_to_F',
                 'paired_p_value']])
paired.to_csv(OUTPUT / 'matched_episodes.csv', index=False)
coverage.to_csv(OUTPUT / 'coverage.csv', index=False)
success.to_csv(OUTPUT / 'success_rates.csv', index=False)
effects.to_csv(OUTPUT / 'paired_effects.csv', index=False)

## Per-suite SR and whole-cohort paired changes

In [ ]:
suite = suite_success_table(paired)
labels = suite.suite.str.removeprefix('libero_')
x = np.arange(len(suite)); width = .2
fig, axes = plt.subplots(2, 1, figsize=(16, 11), constrained_layout=True)
for offset, column, label, color in (
        (-1.5 * width, 'baseline_sr', 'stock baseline', '#4C78A8'),
        (-.5 * width, 'gradient_sr', 'ungated U20 gradient', '#F58518'),
        (.5 * width, 'gate015_sr', 'online gate <= 0.015', '#54A24B'),
        (1.5 * width, 'gate020_sr', 'online gate <= 0.020', '#B279A2')):
    axes[0].bar(x + offset, 100 * suite[column], width, label=label, color=color)
axes[0].set_xticks(x, labels, rotation=40, ha='right')
axes[0].set(ylabel='Success rate (%)', ylim=(0, 105),
            title='LIBERO-PRO success rate on exact matched identities')
axes[0].legend(); axes[0].grid(axis='y', alpha=.2)
for offset, column, label, color in (
        (-.24, 'gradient_sr', 'ungated gradient minus stock', '#F58518'),
        (0, 'gate015_sr', 'gate 0.015 minus stock', '#54A24B'),
        (.24, 'gate020_sr', 'gate 0.020 minus stock', '#B279A2')):
    axes[1].bar(x + offset, 100 * (suite[column] - suite.baseline_sr),
                .23, label=label, color=color)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xticks(x, labels, rotation=40, ha='right')
axes[1].set(ylabel='SR change vs stock (percentage points)',
            title='Whole-matched-cohort paired SR change by suite')
axes[1].legend(); axes[1].grid(axis='y', alpha=.2)
fig.savefig(OUTPUT / 'success_and_suite_deltas.png', dpi=180)
plt.show()
suite.to_csv(OUTPUT / 'suite_success.csv', index=False)

## What the online gate actually accepted

The accept rate is over live action-chunk decisions, not episodes. A rejected candidate returns the exact stock chunk at that boundary.

In [ ]:
telemetry = gate_telemetry_table(paired)
display(telemetry)
telemetry.to_csv(OUTPUT / 'gate_telemetry.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].bar(telemetry.arm, telemetry.gradient_chunk_accept_rate_pct,
            color=['#54A24B', '#B279A2'])
axes[0].set(ylabel='Gradient chunks accepted (%)', ylim=(0, 105),
            title='Online gate acceptance')
axes[0].tick_params(axis='x', rotation=15)
for prefix, label, color in (
        ('gate015', 'cap 0.015', '#54A24B'),
        ('gate020', 'cap 0.020', '#B279A2')):
    values = paired[f'{prefix}_mean_action_rms'].dropna()
    axes[1].hist(values, bins=25, alpha=.55, label=label, color=color)
axes[1].set(xlabel='Mean per-episode first-10 arm-action RMS', ylabel='Episodes',
            title='Candidate displacement distribution')
axes[1].legend()
fig.savefig(OUTPUT / 'gate_acceptance_and_displacement.png', dpi=180)
plt.show()

## Reading the result

The gate is useful only if a gated arm beats both stock and the ungated gradient on the paired SR table. Acceptance statistics explain how restrictive each cap was; they are diagnostics, not additional outcome-tuned thresholds.